In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("gpt-5")
standard_model = init_chat_model("gpt-5-nano")


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [10]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [11]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

I didn’t water it today—I’m not there to do the physical watering. But I can help with it in other ways. What would you like me to do?

- Set a reminder to water it tomorrow or on a schedule.
- Log today’s watering and note the next due date in the care sheet.
- Message Facilities or the office manager to take care of it.

Tell me which option you prefer, and I’ll take care of it.


In [12]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-5-nano-2025-08-07


In [13]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

Roughly every 1–2 years for most houseplants, sooner (6–12 months) if it’s a fast grower. Instead of the calendar, watch for these signs:
- Roots circling the surface or poking out the drainage holes
- Water rushing straight through or the soil drying out much faster than it used to
- The plant getting top‑heavy or growth stalling
- Salt crust on the soil or very compacted mix

Best time to up‑pot is spring. When we do, we’ll move it just 1–2 inches wider with good drainage and fresh mix. I can check the roots at the next watering and let you know if it’s getting tight.


In [14]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-5-2025-08-07
